# Notebook 02: Normalizacion de Transcripciones

Estudio completo de los errores que introduce Whisper y pipeline de normalizacion.

**Objetivos:**
1. Analizar errores sistematicos del ASR (alucinaciones, repeticiones, errores tecnicos)
2. Cuantificar la magnitud de cada tipo de error
3. Construir un pipeline de normalizacion robusto
4. Comparar estadisticas antes/despues
5. Exportar transcripciones normalizadas

**Input:** `artifacts/llm_{id}/audio.json` (transcripciones Whisper crudas)

**Output:** `artifacts/llm_{id}/audio_normalized.json` (transcripciones limpias)

---
## 1. Setup y carga de datos

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter, defaultdict
from datetime import timedelta
from copy import deepcopy
import unicodedata

# Paths
cwd = Path.cwd()
if cwd.name == "notebooks" and cwd.parent.name == "Proyecto_Final_Alejandro":
    PROJECT_ROOT = cwd.parent.parent
elif cwd.name in ("notebooks", "Proyecto_Final_Alejandro"):
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

# Cargar todas las sesiones
sessions = {}
for audio_dir in sorted(ARTIFACTS_DIR.glob("llm_*")):
    audio_file = audio_dir / "audio.json"
    if not audio_file.exists():
        continue
    with open(audio_file) as f:
        data = json.load(f)
    segments = data.get("audio", data).get("segments", [])
    session_id = audio_dir.name
    sessions[session_id] = {
        "path": audio_file,
        "data": data,
        "segments": segments,
        "duration_ms": data.get("audio", data).get("duration_ms", 0),
    }

print(f"Sesiones cargadas: {len(sessions)}")
total_segs = sum(len(s["segments"]) for s in sessions.values())
total_words = sum(len(seg["text"].split()) for s in sessions.values() for seg in s["segments"])
print(f"Total segmentos: {total_segs:,}")
print(f"Total palabras: {total_words:,}")
print()
for sid, s in sessions.items():
    n = len(s["segments"])
    w = sum(len(seg["text"].split()) for seg in s["segments"])
    dur = s["duration_ms"] / 60_000
    print(f"  {sid:10s}: {n:5d} segmentos, {w:6d} palabras, {dur:.0f} min")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#FAFAFA",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "figure.dpi": 100,
})

PALETTE = ["#1B2A4A", "#38BDF8", "#10B981", "#F59E0B", "#8B5CF6", "#EC4899", "#EF4444", "#6366F1"]

# --- Grafico 1: Vision general por sesion ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sids = list(sessions.keys())
seg_counts = [len(sessions[s]["segments"]) for s in sids]
word_counts = [sum(len(seg["text"].split()) for seg in sessions[s]["segments"]) for s in sids]
durations = [sessions[s]["duration_ms"] / 60_000 for s in sids]

# Segmentos por sesion
axes[0].bar(range(len(sids)), seg_counts, color=PALETTE[1], edgecolor="white", linewidth=0.5)
axes[0].set_xticks(range(len(sids)))
axes[0].set_xticklabels([s.replace("llm_", "") for s in sids], rotation=45)
axes[0].set_title("Segmentos por sesion", fontweight="bold")
axes[0].set_ylabel("Segmentos")
for i, v in enumerate(seg_counts):
    axes[0].text(i, v + max(seg_counts)*0.02, str(v), ha="center", fontsize=9)

# Palabras por sesion
axes[1].bar(range(len(sids)), word_counts, color=PALETTE[2], edgecolor="white", linewidth=0.5)
axes[1].set_xticks(range(len(sids)))
axes[1].set_xticklabels([s.replace("llm_", "") for s in sids], rotation=45)
axes[1].set_title("Palabras por sesion", fontweight="bold")
axes[1].set_ylabel("Palabras")
axes[1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

# Duracion por sesion
axes[2].bar(range(len(sids)), durations, color=PALETTE[3], edgecolor="white", linewidth=0.5)
axes[2].set_xticks(range(len(sids)))
axes[2].set_xticklabels([s.replace("llm_", "") for s in sids], rotation=45)
axes[2].set_title("Duracion por sesion (min)", fontweight="bold")
axes[2].set_ylabel("Minutos")

fig.suptitle("Vision general del corpus", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Deteccion de alucinaciones (segmentos repetidos)

Whisper a veces genera segmentos repetidos identicos, especialmente al inicio/fin de las grabaciones o durante silencios largos. Estos son "alucinaciones" del modelo.

In [ ]:
def detect_hallucinations(segments, max_repeat=2):
    """Detecta segmentos repetidos consecutivos (alucinaciones Whisper).
    
    Returns:
        list of dicts con info de cada bloque de repeticiones.
    """
    hallucinations = []
    i = 0
    while i < len(segments):
        text = segments[i]["text"].strip().lower()
        if not text:
            i += 1
            continue
        # Contar repeticiones consecutivas
        j = i + 1
        while j < len(segments) and segments[j]["text"].strip().lower() == text:
            j += 1
        count = j - i
        if count > max_repeat:
            hallucinations.append({
                "text": segments[i]["text"].strip(),
                "count": count,
                "start_idx": i,
                "end_idx": j - 1,
                "start_ms": segments[i]["start_ms"],
                "end_ms": segments[j - 1]["end_ms"],
            })
        i = j
    return hallucinations


print("=" * 70)
print("ANALISIS DE ALUCINACIONES (segmentos repetidos > 2 veces)")
print("=" * 70)

total_hallucinated_segs = 0
all_hallucinations = {}

for sid, s in sessions.items():
    halls = detect_hallucinations(s["segments"])
    all_hallucinations[sid] = halls
    if halls:
        n_segs = sum(h["count"] for h in halls)
        total_hallucinated_segs += n_segs
        print(f"\n{sid}: {len(halls)} bloques de alucinacion ({n_segs} segmentos)")
        for h in halls:
            ts = str(timedelta(milliseconds=h["start_ms"]))[:-3]
            print(f"  [{ts}] \"{h['text'][:60]}\" x{h['count']}")
    else:
        print(f"\n{sid}: Sin alucinaciones detectadas")

pct = total_hallucinated_segs / total_segs * 100 if total_segs else 0
print(f"\nTotal segmentos alucinados: {total_hallucinated_segs} / {total_segs} ({pct:.2f}%)")

In [ ]:
# --- Grafico 2: Alucinaciones por sesion ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

hall_counts = [sum(h["count"] for h in all_hallucinations.get(s, [])) for s in sids]
hall_blocks = [len(all_hallucinations.get(s, [])) for s in sids]

colors = [PALETTE[6] if c > 0 else PALETTE[1] for c in hall_counts]
axes[0].bar(range(len(sids)), hall_counts, color=colors, edgecolor="white", linewidth=0.5)
axes[0].set_xticks(range(len(sids)))
axes[0].set_xticklabels([s.replace("llm_", "") for s in sids], rotation=45)
axes[0].set_title("Segmentos alucinados por sesion", fontweight="bold")
axes[0].set_ylabel("Segmentos repetidos")
for i, v in enumerate(hall_counts):
    if v > 0:
        axes[0].text(i, v + max(hall_counts)*0.03, str(v), ha="center", fontsize=9, color=PALETTE[6])

# Proporcion alucinados vs normales (stacked)
normal_counts = [len(sessions[s]["segments"]) - hall_counts[i] for i, s in enumerate(sids)]
axes[1].bar(range(len(sids)), normal_counts, color=PALETTE[1], label="Normal", edgecolor="white")
axes[1].bar(range(len(sids)), hall_counts, bottom=normal_counts, color=PALETTE[6], label="Alucinados", edgecolor="white")
axes[1].set_xticks(range(len(sids)))
axes[1].set_xticklabels([s.replace("llm_", "") for s in sids], rotation=45)
axes[1].set_title("Proporcion de alucinaciones", fontweight="bold")
axes[1].set_ylabel("Segmentos")
axes[1].legend(loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()

---
## 3. Deteccion de segmentos cuasi-duplicados

Ademas de repeticiones exactas, Whisper puede generar segmentos muy similares (misma frase con pequenas variaciones). Usamos similitud de Jaccard sobre tokens.

In [ ]:
def jaccard_similarity(a, b):
    """Similitud de Jaccard entre dos strings (basada en tokens)."""
    tokens_a = set(a.lower().split())
    tokens_b = set(b.lower().split())
    if not tokens_a or not tokens_b:
        return 0.0
    intersection = tokens_a & tokens_b
    union = tokens_a | tokens_b
    return len(intersection) / len(union)


def detect_near_duplicates(segments, threshold=0.85, window=5):
    """Detecta pares de segmentos cuasi-duplicados dentro de una ventana."""
    dupes = []
    for i in range(len(segments)):
        text_i = segments[i]["text"].strip()
        if len(text_i.split()) < 4:
            continue
        for j in range(i + 1, min(i + window, len(segments))):
            text_j = segments[j]["text"].strip()
            sim = jaccard_similarity(text_i, text_j)
            if sim >= threshold and text_i.lower() != text_j.lower():
                dupes.append({
                    "idx_a": i, "idx_b": j,
                    "text_a": text_i, "text_b": text_j,
                    "similarity": round(sim, 3),
                    "start_ms": segments[i]["start_ms"],
                })
    return dupes


print("=" * 70)
print("CUASI-DUPLICADOS (Jaccard >= 0.85, ventana 5 segmentos)")
print("=" * 70)

total_near_dupes = 0
all_near_dupes = {}

for sid, s in sessions.items():
    dupes = detect_near_duplicates(s["segments"])
    all_near_dupes[sid] = dupes
    total_near_dupes += len(dupes)
    if dupes:
        print(f"\n{sid}: {len(dupes)} pares cuasi-duplicados")
        for d in dupes[:3]:
            print(f"  sim={d['similarity']:.2f}")
            print(f"    A: \"{d['text_a'][:70]}\"")
            print(f"    B: \"{d['text_b'][:70]}\"")
    else:
        print(f"\n{sid}: Sin cuasi-duplicados")

print(f"\nTotal pares cuasi-duplicados: {total_near_dupes}")

---
## 4. Analisis de errores en terminos tecnicos

Whisper confunde frecuentemente terminos tecnicos en ingles cuando el contexto es espanol. Analizamos la frecuencia de errores conocidos y descubrimos nuevos.

In [ ]:
# Diccionario expandido de errores ASR conocidos
# Formato: {"error": "correccion"}
KNOWN_ASR_ERRORS = {
    # ML/DL core terms
    "back of works": "Bag of Words",
    "bag of works": "Bag of Words",
    "back of words": "Bag of Words",
    "bac of words": "Bag of Words",
    "bags of words": "Bag of Words",
    "overfiting": "overfitting",
    "over fitting": "overfitting",
    "over-fitting": "overfitting",
    "underfiting": "underfitting",
    "under fitting": "underfitting",
    "under-fitting": "underfitting",
    "gradient decent": "gradient descent",
    "gradien descent": "gradient descent",
    "back propagation": "backpropagation",
    "data aumentation": "data augmentation",
    "data augmentacion": "data augmentation",
    "fine tunning": "fine-tuning",
    "fine tuning": "fine-tuning",
    "fine tunnig": "fine-tuning",
    "transfer lerning": "transfer learning",
    "transferlearning": "transfer learning",
    "cross validacion": "cross-validation",
    "cross validation": "cross-validation",
    "hyperparametro": "hiperparametro",
    "embedings": "embeddings",
    "embeding": "embedding",
    "tokenizacion": "tokenizacion",
    "learnin rate": "learning rate",
    "neur network": "neural network",
    "neuronal network": "neural network",
    # Frameworks
    "pytorch": "PyTorch",
    "pytoch": "PyTorch",
    "tensorflow": "TensorFlow",
    "scikit learn": "scikit-learn",
    # Architectures
    "self-atention": "self-attention",
    "self atention": "self-attention",
    # NLP terms
    "word 2 vec": "Word2Vec",
    "tf idf": "TF-IDF",
    "tfidf": "TF-IDF",
    # Spanish ASR confusions
    "datos estupurados": "datos estructurados",
    "datos estrupturados": "datos estructurados",
    "descalaridad": "escalabilidad",
    "mas net": "ImageNet",
    "image net": "ImageNet",
}

# Buscar ocurrencias en todas las transcripciones
error_counts = Counter()
error_examples = defaultdict(list)

for sid, s in sessions.items():
    for seg in s["segments"]:
        text_lower = seg["text"].lower()
        for error in KNOWN_ASR_ERRORS:
            if error in text_lower:
                error_counts[error] += 1
                if len(error_examples[error]) < 2:
                    ts = str(timedelta(milliseconds=seg["start_ms"]))[:-3]
                    error_examples[error].append(f"[{sid} {ts}] {seg['text'][:80]}")

print("=" * 70)
print("ERRORES ASR CONOCIDOS ENCONTRADOS")
print("=" * 70)

if error_counts:
    for error, count in error_counts.most_common():
        canonical = KNOWN_ASR_ERRORS[error]
        print(f"\n  \"{error}\" -> \"{canonical}\"  ({count} ocurrencias)")
        for ex in error_examples[error]:
            print(f"    {ex}")
else:
    print("No se encontraron errores ASR conocidos en los segmentos.")

print(f"\nTotal errores detectados: {sum(error_counts.values())}")
print(f"Tipos de error distintos: {len(error_counts)}")

---
## 5. Analisis de frecuencia de palabras y n-gramas

Buscamos patrones sospechosos: palabras muy frecuentes que podrian ser errores sistematicos, n-gramas repetidos que delaten alucinaciones parciales.

In [ ]:
# Frecuencia de palabras globales
word_counter = Counter()
bigram_counter = Counter()

for sid, s in sessions.items():
    for seg in s["segments"]:
        words = seg["text"].lower().split()
        word_counter.update(words)
        for i in range(len(words) - 1):
            bigram_counter[f"{words[i]} {words[i+1]}"] += 1

print("TOP 30 PALABRAS MAS FRECUENTES")
print("-" * 40)
for word, count in word_counter.most_common(30):
    print(f"  {word:25s} {count:6d}")

print(f"\nTOP 20 BIGRAMAS MAS FRECUENTES")
print("-" * 40)
for bigram, count in bigram_counter.most_common(20):
    print(f"  {bigram:35s} {count:6d}")

In [ ]:
# --- Grafico 3: Ley de Zipf (log-log de frecuencia de palabras) ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Zipf plot
ranks = np.arange(1, len(word_counter) + 1)
freqs = [count for _, count in word_counter.most_common()]

axes[0].loglog(ranks, freqs, color=PALETTE[0], linewidth=1.5, alpha=0.8)
# Linea de referencia Zipf ideal (f = C / r)
C = freqs[0]
zipf_ideal = C / ranks
axes[0].loglog(ranks, zipf_ideal, '--', color=PALETTE[6], alpha=0.5, label="Zipf ideal (1/r)")
axes[0].set_xlabel("Rango (log)")
axes[0].set_ylabel("Frecuencia (log)")
axes[0].set_title("Ley de Zipf — Distribucion de palabras", fontweight="bold")
axes[0].legend()

# Top 25 palabras (horizontal bar)
top_n = 25
top_words = word_counter.most_common(top_n)
words_list = [w for w, _ in reversed(top_words)]
counts_list = [c for _, c in reversed(top_words)]

colors_bar = [PALETTE[1] if w not in {"que", "de", "el", "la", "en", "y", "es", "se", "no", "lo", "un", "por", "con", "los", "las"} else PALETTE[0] for w in words_list]
axes[1].barh(range(top_n), counts_list, color=colors_bar, edgecolor="white", linewidth=0.5)
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(words_list, fontsize=10)
axes[1].set_xlabel("Frecuencia")
axes[1].set_title(f"Top {top_n} palabras mas frecuentes", fontweight="bold")
axes[1].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else f"{x:.0f}"))

plt.tight_layout()
plt.show()

# --- Grafico 4: Top 20 bigramas ---
fig, ax = plt.subplots(figsize=(12, 5))
top_bi = 20
top_bigrams = bigram_counter.most_common(top_bi)
bi_labels = [b for b, _ in reversed(top_bigrams)]
bi_counts = [c for _, c in reversed(top_bigrams)]

ax.barh(range(top_bi), bi_counts, color=PALETTE[4], edgecolor="white", linewidth=0.5)
ax.set_yticks(range(top_bi))
ax.set_yticklabels(bi_labels, fontsize=10)
ax.set_xlabel("Frecuencia")
ax.set_title(f"Top {top_bi} bigramas mas frecuentes", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Detectar posibles errores no catalogados:
# Palabras en ingles que aparecen con variantes sospechosas

ENGLISH_TECH_TERMS = {
    "network", "learning", "training", "model", "layer", "batch",
    "weight", "bias", "feature", "embedding", "attention", "encoder",
    "decoder", "transformer", "token", "gradient", "loss", "epoch",
    "dropout", "pooling", "kernel", "filter", "stride", "padding",
    "softmax", "sigmoid", "relu", "activation", "backpropagation",
    "dataset", "preprocessing", "pipeline", "inference", "benchmark",
    "accuracy", "precision", "recall", "overfitting", "underfitting",
    "regularization", "normalization", "optimization", "convergence",
}

# Buscar variantes cercanas (edit distance = 1-2)
def levenshtein(s1, s2):
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = prev_row[j + 1] + 1
            deletions = curr_row[j] + 1
            substitutions = prev_row[j] + (c1 != c2)
            curr_row.append(min(insertions, deletions, substitutions))
        prev_row = curr_row
    return prev_row[-1]


# Palabras que estan a distancia 1-2 de un termino tecnico conocido
suspicious = []
checked = set()

for word, count in word_counter.most_common(2000):
    if word in checked or len(word) < 5 or count < 3:
        continue
    for tech in ENGLISH_TECH_TERMS:
        dist = levenshtein(word, tech)
        if 0 < dist <= 2 and word != tech:
            suspicious.append((word, tech, dist, count))
            checked.add(word)
            break

if suspicious:
    print("POSIBLES ERRORES ASR NO CATALOGADOS")
    print("-" * 60)
    for word, tech, dist, count in sorted(suspicious, key=lambda x: -x[3]):
        print(f"  \"{word}\" -> \"{tech}\"? (dist={dist}, aparece {count} veces)")
else:
    print("No se detectaron errores adicionales por distancia de edicion.")

---
## 6. Analisis de mezcla de idiomas (ES/EN)

Las clases mezclan espanol e ingles tecnico. Whisper a veces cambia de idioma mid-sentence, produciendo transcripciones inconsistentes.

In [ ]:
# Heuristica simple: detectar segmentos con alta proporcion de palabras en ingles
SPANISH_STOPWORDS = {
    "el", "la", "los", "las", "un", "una", "de", "del", "en", "y", "que",
    "es", "se", "no", "lo", "por", "con", "para", "como", "pero", "mas",
    "su", "al", "le", "ya", "o", "si", "me", "ha", "este", "esta", "son",
    "tiene", "hay", "ser", "muy", "nos", "ese", "eso", "aqui", "vamos",
    "pues", "vale", "sea", "sino", "cada", "entonces", "todo", "tenemos",
}

ENGLISH_COMMON = {
    "the", "is", "are", "was", "were", "this", "that", "with", "for",
    "and", "not", "you", "have", "from", "they", "been", "has", "its",
    "can", "will", "would", "should", "could", "just", "like", "some",
}


def classify_language(text):
    """Clasifica un segmento como ES, EN, o MIXED."""
    words = re.findall(r'[a-zA-ZáéíóúñÁÉÍÓÚÑü]+', text.lower())
    if len(words) < 3:
        return "SHORT"
    es_count = sum(1 for w in words if w in SPANISH_STOPWORDS)
    en_count = sum(1 for w in words if w in ENGLISH_COMMON)
    es_ratio = es_count / len(words)
    en_ratio = en_count / len(words)
    if es_ratio > 0.2 and en_ratio > 0.15:
        return "MIXED"
    elif en_ratio > 0.3:
        return "EN"
    else:
        return "ES"


lang_stats = Counter()
mixed_examples = []
en_examples = []

for sid, s in sessions.items():
    for seg in s["segments"]:
        lang = classify_language(seg["text"])
        lang_stats[lang] += 1
        if lang == "MIXED" and len(mixed_examples) < 5:
            ts = str(timedelta(milliseconds=seg["start_ms"]))[:-3]
            mixed_examples.append(f"[{sid} {ts}] {seg['text'][:100]}")
        elif lang == "EN" and len(en_examples) < 5:
            ts = str(timedelta(milliseconds=seg["start_ms"]))[:-3]
            en_examples.append(f"[{sid} {ts}] {seg['text'][:100]}")

print("DISTRIBUCION DE IDIOMA POR SEGMENTO")
print("-" * 40)
for lang, count in lang_stats.most_common():
    pct = count / sum(lang_stats.values()) * 100
    bar = '#' * int(pct / 2)
    print(f"  {lang:6s}: {count:5d} ({pct:5.1f}%) {bar}")

if mixed_examples:
    print("\nEjemplos MIXED (ES+EN en mismo segmento):")
    for ex in mixed_examples:
        print(f"  {ex}")

if en_examples:
    print("\nEjemplos EN (segmentos predominantemente en ingles):")
    for ex in en_examples:
        print(f"  {ex}")

In [ ]:
# --- Grafico 5: Distribucion de idioma (pie + bar por sesion) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
lang_labels = [k for k, _ in lang_stats.most_common()]
lang_values = [v for _, v in lang_stats.most_common()]
lang_colors = {"ES": PALETTE[1], "EN": PALETTE[6], "MIXED": PALETTE[3], "SHORT": "#CCCCCC"}
pie_colors = [lang_colors.get(l, PALETTE[4]) for l in lang_labels]

wedges, texts, autotexts = axes[0].pie(
    lang_values, labels=lang_labels, autopct="%1.1f%%",
    colors=pie_colors, startangle=90, textprops={"fontsize": 11})
for t in autotexts:
    t.set_fontsize(10)
    t.set_fontweight("bold")
axes[0].set_title("Distribucion de idioma por segmento", fontweight="bold")

# Bar chart por sesion (stacked)
lang_per_session = {}
for sid, s in sessions.items():
    counts = Counter()
    for seg in s["segments"]:
        counts[classify_language(seg["text"])] += 1
    lang_per_session[sid] = counts

x = np.arange(len(sids))
bottom = np.zeros(len(sids))
for lang in ["ES", "EN", "MIXED", "SHORT"]:
    vals = [lang_per_session[s].get(lang, 0) for s in sids]
    axes[1].bar(x, vals, bottom=bottom, label=lang, color=lang_colors.get(lang, PALETTE[4]),
                edgecolor="white", linewidth=0.5)
    bottom += np.array(vals)

axes[1].set_xticks(x)
axes[1].set_xticklabels([s.replace("llm_", "") for s in sids], rotation=45)
axes[1].set_title("Idioma por sesion (stacked)", fontweight="bold")
axes[1].set_ylabel("Segmentos")
axes[1].legend(loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()

---
## 7. Analisis de patrones de puntuacion y capitalizacion

Whisper es inconsistente con la puntuacion y capitalizacion, especialmente con terminos tecnicos.

In [ ]:
# Analisis de puntuacion
punct_stats = {
    "sin_punto_final": 0,
    "con_punto_final": 0,
    "con_interrogacion": 0,
    "con_exclamacion": 0,
    "con_coma": 0,
    "con_puntos_suspensivos": 0,
    "vacio_o_muy_corto": 0,
}

# Capitalizacion de terminos tecnicos
tech_capitalization = defaultdict(Counter)
TECH_TERMS_TO_CHECK = [
    "bag of words", "neural network", "machine learning", "deep learning",
    "random forest", "decision tree", "natural language",
    "gradient descent", "back propagation", "cross validation",
]

for sid, s in sessions.items():
    for seg in s["segments"]:
        text = seg["text"].strip()
        if len(text) < 3:
            punct_stats["vacio_o_muy_corto"] += 1
            continue
        if text.endswith("."):
            punct_stats["con_punto_final"] += 1
        elif text.endswith("?"):
            punct_stats["con_interrogacion"] += 1
        elif text.endswith("!"):
            punct_stats["con_exclamacion"] += 1
        elif text.endswith("..."):
            punct_stats["con_puntos_suspensivos"] += 1
        else:
            punct_stats["sin_punto_final"] += 1
        if "," in text:
            punct_stats["con_coma"] += 1
        
        # Capitalizacion
        text_lower = text.lower()
        for term in TECH_TERMS_TO_CHECK:
            if term in text_lower:
                # Encontrar la variante exacta usada
                idx = text_lower.index(term)
                original = text[idx:idx+len(term)]
                tech_capitalization[term][original] += 1

print("ESTADISTICAS DE PUNTUACION")
print("-" * 40)
for stat, count in sorted(punct_stats.items(), key=lambda x: -x[1]):
    pct = count / total_segs * 100
    print(f"  {stat:30s}: {count:5d} ({pct:.1f}%)")

print("\nVARIANTES DE CAPITALIZACION (terminos tecnicos)")
print("-" * 50)
for term, variants in sorted(tech_capitalization.items()):
    if len(variants) > 1:
        print(f"  \"{term}\":")
        for variant, count in variants.most_common():
            print(f"    \"{variant}\": {count}x")

In [ ]:
# --- Grafico 6: Puntuacion + histograma de longitud de segmentos ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Puntuacion bar chart
punct_labels = list(punct_stats.keys())
punct_values = list(punct_stats.values())
bar_colors = [PALETTE[i % len(PALETTE)] for i in range(len(punct_labels))]

axes[0].barh(range(len(punct_labels)), punct_values, color=bar_colors, edgecolor="white", linewidth=0.5)
axes[0].set_yticks(range(len(punct_labels)))
axes[0].set_yticklabels([l.replace("_", " ") for l in punct_labels], fontsize=10)
axes[0].set_xlabel("Segmentos")
axes[0].set_title("Patrones de puntuacion", fontweight="bold")

# Histograma de longitud de segmentos (palabras)
all_lengths = []
for s in sessions.values():
    for seg in s["segments"]:
        all_lengths.append(len(seg["text"].split()))

axes[1].hist(all_lengths, bins=50, color=PALETTE[1], edgecolor="white", linewidth=0.5, alpha=0.85)
axes[1].axvline(np.median(all_lengths), color=PALETTE[6], linestyle="--", linewidth=2,
                label=f"Mediana: {np.median(all_lengths):.0f} palabras")
axes[1].axvline(np.mean(all_lengths), color=PALETTE[3], linestyle="--", linewidth=2,
                label=f"Media: {np.mean(all_lengths):.1f} palabras")
axes[1].set_xlabel("Palabras por segmento")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribucion de longitud de segmentos", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 8. Analisis de segmentos problematicos

Segmentos muy cortos (< 3 palabras), muy largos (> 100 palabras), o con caracteres extranhos.

In [ ]:
short_segs = []
long_segs = []
weird_chars = []

# Caracteres no esperados en espanol/ingles
NORMAL_CHARS = set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ'
                   'áéíóúÁÉÍÓÚñÑüÜ0123456789 .,;:!?\'-"()[]{}@#$%&*/+=<>\n\t')

for sid, s in sessions.items():
    for i, seg in enumerate(s["segments"]):
        text = seg["text"].strip()
        words = text.split()
        
        if len(words) <= 1 and len(text) > 0:
            short_segs.append((sid, i, seg["start_ms"], text))
        elif len(words) > 80:
            long_segs.append((sid, i, seg["start_ms"], len(words), text[:100]))
        
        odd = set(text) - NORMAL_CHARS
        if odd:
            weird_chars.append((sid, i, odd, text[:60]))

print(f"SEGMENTOS PROBLEMATICOS")
print(f"-" * 40)
print(f"  Muy cortos (1 palabra): {len(short_segs)}")
print(f"  Muy largos (>80 palabras): {len(long_segs)}")
print(f"  Con caracteres extraños: {len(weird_chars)}")

if short_segs:
    print(f"\nEjemplos de segmentos muy cortos:")
    for sid, idx, ms, text in short_segs[:10]:
        ts = str(timedelta(milliseconds=ms))[:-3]
        print(f"  [{sid} {ts}] \"{text}\"")

if long_segs:
    print(f"\nSegmentos muy largos:")
    for sid, idx, ms, nw, text in long_segs[:5]:
        ts = str(timedelta(milliseconds=ms))[:-3]
        print(f"  [{sid} {ts}] {nw} palabras: \"{text}...\"")

if weird_chars:
    print(f"\nCaracteres extraños encontrados:")
    all_weird = set()
    for _, _, odd, _ in weird_chars:
        all_weird |= odd
    print(f"  Caracteres: {all_weird}")
    for sid, idx, odd, text in weird_chars[:5]:
        print(f"  [{sid}] \"{text}\" (chars: {odd})")

In [ ]:
# --- Grafico 7: Boxplot de longitud por sesion + segmentos anomalos ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot de palabras por segmento, por sesion
lengths_by_session = []
labels_box = []
for sid in sids:
    lens = [len(seg["text"].split()) for seg in sessions[sid]["segments"]]
    lengths_by_session.append(lens)
    labels_box.append(sid.replace("llm_", ""))

bp = axes[0].boxplot(lengths_by_session, labels=labels_box, patch_artist=True,
                     medianprops=dict(color=PALETTE[6], linewidth=2),
                     flierprops=dict(marker=".", markersize=3, alpha=0.3))
for i, patch in enumerate(bp["boxes"]):
    patch.set_facecolor(PALETTE[i % len(PALETTE)])
    patch.set_alpha(0.6)
axes[0].set_xlabel("Sesion")
axes[0].set_ylabel("Palabras por segmento")
axes[0].set_title("Distribucion de longitud por sesion", fontweight="bold")
axes[0].tick_params(axis="x", rotation=45)

# Segmentos anomalos: scatter (cortos vs largos)
issue_types = ["Muy cortos\n(1 palabra)", "Muy largos\n(>80 pal.)", "Chars\nextranos"]
issue_counts = [len(short_segs), len(long_segs), len(weird_chars)]
bar_colors = [PALETTE[3], PALETTE[6], PALETTE[5]]

axes[1].bar(issue_types, issue_counts, color=bar_colors, edgecolor="white", linewidth=0.5, width=0.5)
axes[1].set_ylabel("Segmentos")
axes[1].set_title("Segmentos anomalos", fontweight="bold")
for i, v in enumerate(issue_counts):
    axes[1].text(i, v + max(issue_counts)*0.03, str(v), ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

---
## 9. Resumen del analisis

Consolidamos todos los hallazgos antes de definir el pipeline de normalizacion.

In [ ]:
print("=" * 70)
print("RESUMEN DEL ANALISIS DE TRANSCRIPCIONES")
print("=" * 70)
print(f"\nDatos:")
print(f"  Sesiones: {len(sessions)}")
print(f"  Segmentos totales: {total_segs:,}")
print(f"  Palabras totales: {total_words:,}")

print(f"\nProblemas detectados:")
print(f"  1. Alucinaciones (repeticiones >2x): {total_hallucinated_segs} segmentos")
print(f"  2. Cuasi-duplicados (Jaccard>=0.85): {total_near_dupes} pares")
print(f"  3. Errores ASR en terminos tecnicos: {sum(error_counts.values())} ocurrencias ({len(error_counts)} tipos)")
print(f"  4. Segmentos muy cortos (1 palabra): {len(short_segs)}")
print(f"  5. Segmentos muy largos (>80 pal.): {len(long_segs)}")
print(f"  6. Caracteres extraños: {len(weird_chars)} segmentos")

total_issues = total_hallucinated_segs + total_near_dupes + sum(error_counts.values()) + len(short_segs) + len(weird_chars)
print(f"\nTotal issues detectados: ~{total_issues}")
print(f"Impacto estimado: {total_issues / total_segs * 100:.1f}% de segmentos afectados")

In [ ]:
# --- Grafico 8: Resumen de problemas detectados (pie + bar) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart de tipos de problema
issue_labels = ["Alucinaciones", "Cuasi-duplicados", "Errores ASR", "Muy cortos", "Chars extranos"]
issue_values = [total_hallucinated_segs, total_near_dupes, sum(error_counts.values()), len(short_segs), len(weird_chars)]
issue_colors = [PALETTE[6], PALETTE[5], PALETTE[3], PALETTE[4], PALETTE[7]]

# Filtrar valores 0
filtered = [(l, v, c) for l, v, c in zip(issue_labels, issue_values, issue_colors) if v > 0]
if filtered:
    fl, fv, fc = zip(*filtered)
    wedges, texts, autotexts = axes[0].pie(fv, labels=fl, autopct="%1.1f%%",
                                            colors=fc, startangle=90, textprops={"fontsize": 10})
    for t in autotexts:
        t.set_fontsize(9)
        t.set_fontweight("bold")
axes[0].set_title("Distribucion de problemas detectados", fontweight="bold")

# Impacto por sesion (total issues)
issues_per_session = []
for sid in sids:
    n = (sum(h["count"] for h in all_hallucinations.get(sid, []))
         + len(all_near_dupes.get(sid, []))
         + sum(1 for seg in sessions[sid]["segments"] if len(seg["text"].split()) <= 1))
    issues_per_session.append(n)

pct_affected = [issues_per_session[i] / len(sessions[sids[i]]["segments"]) * 100
                for i in range(len(sids))]

bars = axes[1].bar(range(len(sids)), pct_affected, color=PALETTE[1], edgecolor="white", linewidth=0.5)
for i, (bar, pct) in enumerate(zip(bars, pct_affected)):
    if pct > 5:
        bar.set_color(PALETTE[6])
    elif pct > 2:
        bar.set_color(PALETTE[3])
axes[1].set_xticks(range(len(sids)))
axes[1].set_xticklabels([s.replace("llm_", "") for s in sids], rotation=45)
axes[1].set_ylabel("% segmentos afectados")
axes[1].set_title("Impacto por sesion (%)", fontweight="bold")
axes[1].axhline(y=5, color=PALETTE[6], linestyle="--", alpha=0.5, label="Umbral alto (5%)")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 10. Pipeline de normalizacion

Definimos las reglas de normalizacion en orden de aplicacion:

| Paso | Regla | Accion |
|------|-------|--------|
| 1 | Eliminar alucinaciones | Reducir bloques repetidos a 1 instancia |
| 2 | Merge cuasi-duplicados | Mantener el segmento mas largo del par |
| 3 | Correccion de terminos tecnicos | Diccionario KNOWN_ASR_ERRORS |
| 4 | Normalizacion Unicode | NFKC, espacios multiples |
| 5 | Limpieza de caracteres | Eliminar caracteres no esperados |
| 6 | Normalizacion de puntuacion | Puntos finales consistentes |

In [ ]:
def normalize_transcription(segments):
    """Pipeline completo de normalizacion.
    
    Returns:
        (normalized_segments, stats_dict)
    """
    stats = {
        "input_segments": len(segments),
        "hallucinations_removed": 0,
        "near_dupes_merged": 0,
        "terms_corrected": 0,
        "unicode_normalized": 0,
        "chars_cleaned": 0,
        "punct_normalized": 0,
    }
    
    # --- Paso 1: Eliminar alucinaciones ---
    deduped = []
    i = 0
    while i < len(segments):
        text = segments[i]["text"].strip().lower()
        j = i + 1
        while j < len(segments) and segments[j]["text"].strip().lower() == text:
            j += 1
        count = j - i
        if count > 2:
            # Mantener solo el primero, ajustar end_ms al ultimo
            merged = deepcopy(segments[i])
            merged["end_ms"] = segments[j - 1]["end_ms"]
            deduped.append(merged)
            stats["hallucinations_removed"] += count - 1
        else:
            for k in range(i, j):
                deduped.append(deepcopy(segments[k]))
        i = j
    
    # --- Paso 2: Merge cuasi-duplicados ---
    merged_segs = []
    skip = set()
    for i in range(len(deduped)):
        if i in skip:
            continue
        text_i = deduped[i]["text"].strip()
        if len(text_i.split()) < 4:
            merged_segs.append(deduped[i])
            continue
        best = deduped[i]
        for j in range(i + 1, min(i + 5, len(deduped))):
            if j in skip:
                continue
            text_j = deduped[j]["text"].strip()
            sim = jaccard_similarity(text_i, text_j)
            if sim >= 0.85 and text_i.lower() != text_j.lower():
                # Mantener el mas largo
                if len(text_j) > len(best["text"]):
                    best = deepcopy(deduped[j])
                    best["start_ms"] = min(best["start_ms"], deduped[i]["start_ms"])
                best["end_ms"] = max(best["end_ms"], deduped[j]["end_ms"])
                skip.add(j)
                stats["near_dupes_merged"] += 1
        merged_segs.append(best)
    
    # --- Paso 3: Correccion de terminos tecnicos ---
    for seg in merged_segs:
        original = seg["text"]
        corrected = original
        for error, canonical in KNOWN_ASR_ERRORS.items():
            pattern = re.compile(re.escape(error), re.IGNORECASE)
            if pattern.search(corrected):
                corrected = pattern.sub(canonical, corrected)
                stats["terms_corrected"] += 1
        seg["text"] = corrected
    
    # --- Paso 4: Normalizacion Unicode ---
    for seg in merged_segs:
        original = seg["text"]
        normalized = unicodedata.normalize("NFKC", original)
        # Colapsar espacios multiples
        normalized = re.sub(r' {2,}', ' ', normalized).strip()
        if normalized != original:
            stats["unicode_normalized"] += 1
        seg["text"] = normalized
    
    # --- Paso 5: Limpieza de caracteres extraños ---
    for seg in merged_segs:
        original = seg["text"]
        # Mantener alfanumericos, puntuacion basica, acentos
        cleaned = re.sub(r'[^\w\s.,;:!?\'\-"()\[\]áéíóúÁÉÍÓÚñÑüÜ]', '', original)
        if cleaned != original:
            stats["chars_cleaned"] += 1
        seg["text"] = cleaned
    
    # --- Paso 6: Normalizacion de puntuacion ---
    for seg in merged_segs:
        text = seg["text"].strip()
        if not text:
            continue
        # Asegurar que empieza en mayuscula
        if text[0].islower():
            text = text[0].upper() + text[1:]
        # Si no termina en puntuacion, agregar punto
        if text[-1] not in '.?!…':
            text += '.'
            stats["punct_normalized"] += 1
        seg["text"] = text
    
    # Eliminar segmentos vacios
    final = [seg for seg in merged_segs if seg["text"].strip()]
    stats["output_segments"] = len(final)
    stats["segments_removed"] = stats["input_segments"] - len(final)
    
    return final, stats


print("Pipeline de normalizacion definido.")
print("6 pasos: alucinaciones -> cuasi-duplicados -> terminos -> unicode -> caracteres -> puntuacion")

---
## 11. Aplicar normalizacion a todas las sesiones

In [ ]:
normalized_sessions = {}
all_stats = {}

print("=" * 70)
print("APLICANDO NORMALIZACION")
print("=" * 70)

for sid, s in sessions.items():
    norm_segs, stats = normalize_transcription(s["segments"])
    normalized_sessions[sid] = norm_segs
    all_stats[sid] = stats
    
    removed = stats["input_segments"] - stats["output_segments"]
    print(f"\n{sid}:")
    print(f"  Segmentos: {stats['input_segments']} -> {stats['output_segments']} (-{removed})")
    print(f"  Alucinaciones eliminadas: {stats['hallucinations_removed']}")
    print(f"  Cuasi-duplicados fusionados: {stats['near_dupes_merged']}")
    print(f"  Terminos corregidos: {stats['terms_corrected']}")
    print(f"  Unicode normalizados: {stats['unicode_normalized']}")
    print(f"  Caracteres limpiados: {stats['chars_cleaned']}")
    print(f"  Puntuacion normalizada: {stats['punct_normalized']}")

---
## 12. Comparacion antes / despues

In [ ]:
print("=" * 70)
print("COMPARACION ANTES / DESPUES")
print("=" * 70)

total_before = sum(s["input_segments"] for s in all_stats.values())
total_after = sum(s["output_segments"] for s in all_stats.values())
total_hallu = sum(s["hallucinations_removed"] for s in all_stats.values())
total_dupes = sum(s["near_dupes_merged"] for s in all_stats.values())
total_terms = sum(s["terms_corrected"] for s in all_stats.values())
total_unicode = sum(s["unicode_normalized"] for s in all_stats.values())
total_chars = sum(s["chars_cleaned"] for s in all_stats.values())
total_punct = sum(s["punct_normalized"] for s in all_stats.values())

# Palabras antes y despues
words_before = sum(len(seg["text"].split()) for s in sessions.values() for seg in s["segments"])
words_after = sum(len(seg["text"].split()) for segs in normalized_sessions.values() for seg in segs)

print(f"\n{'Metrica':<35s} {'Antes':>10s} {'Despues':>10s} {'Delta':>10s}")
print("-" * 70)
print(f"{'Segmentos':<35s} {total_before:>10,d} {total_after:>10,d} {total_after - total_before:>+10,d}")
print(f"{'Palabras':<35s} {words_before:>10,d} {words_after:>10,d} {words_after - words_before:>+10,d}")
print(f"")
print(f"{'Acciones de normalizacion':<35s} {'':>10s} {'Total':>10s}")
print("-" * 70)
print(f"{'Alucinaciones eliminadas':<35s} {'':>10s} {total_hallu:>10,d}")
print(f"{'Cuasi-duplicados fusionados':<35s} {'':>10s} {total_dupes:>10,d}")
print(f"{'Terminos tecnicos corregidos':<35s} {'':>10s} {total_terms:>10,d}")
print(f"{'Unicode normalizados':<35s} {'':>10s} {total_unicode:>10,d}")
print(f"{'Caracteres limpiados':<35s} {'':>10s} {total_chars:>10,d}")
print(f"{'Puntuacion normalizada':<35s} {'':>10s} {total_punct:>10,d}")

total_actions = total_hallu + total_dupes + total_terms + total_unicode + total_chars + total_punct
print(f"\nTotal acciones aplicadas: {total_actions:,d}")

In [ ]:
# --- Grafico 9: Comparacion antes/despues ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Segmentos antes vs despues por sesion (grouped bar)
x = np.arange(len(sids))
width = 0.35
before_segs = [all_stats[s]["input_segments"] for s in sids]
after_segs = [all_stats[s]["output_segments"] for s in sids]

axes[0].bar(x - width/2, before_segs, width, label="Antes", color=PALETTE[6], alpha=0.7, edgecolor="white")
axes[0].bar(x + width/2, after_segs, width, label="Despues", color=PALETTE[2], alpha=0.7, edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels([s.replace("llm_", "") for s in sids], rotation=45)
axes[0].set_ylabel("Segmentos")
axes[0].set_title("Segmentos antes vs despues", fontweight="bold")
axes[0].legend()

# Acciones de normalizacion por tipo (stacked bar)
action_types = ["Alucinaciones", "Duplicados", "Terminos", "Unicode", "Chars", "Puntuacion"]
action_colors = [PALETTE[6], PALETTE[5], PALETTE[3], PALETTE[4], PALETTE[7], PALETTE[1]]

bottom = np.zeros(len(sids))
for action, color, key in zip(action_types, action_colors,
    ["hallucinations_removed", "near_dupes_merged", "terms_corrected",
     "unicode_normalized", "chars_cleaned", "punct_normalized"]):
    vals = [all_stats[s][key] for s in sids]
    if sum(vals) > 0:
        axes[1].bar(x, vals, bottom=bottom, label=action, color=color, edgecolor="white", linewidth=0.5)
        bottom += np.array(vals)

axes[1].set_xticks(x)
axes[1].set_xticklabels([s.replace("llm_", "") for s in sids], rotation=45)
axes[1].set_ylabel("Acciones")
axes[1].set_title("Acciones de normalizacion por sesion", fontweight="bold")
axes[1].legend(fontsize=8, loc="upper right")

# Palabras antes vs despues (total)
words_b = [sum(len(seg["text"].split()) for seg in sessions[s]["segments"]) for s in sids]
words_a = [sum(len(seg["text"].split()) for seg in normalized_sessions[s]) for s in sids]

axes[2].bar(x - width/2, words_b, width, label="Antes", color=PALETTE[6], alpha=0.7, edgecolor="white")
axes[2].bar(x + width/2, words_a, width, label="Despues", color=PALETTE[2], alpha=0.7, edgecolor="white")
axes[2].set_xticks(x)
axes[2].set_xticklabels([s.replace("llm_", "") for s in sids], rotation=45)
axes[2].set_ylabel("Palabras")
axes[2].set_title("Palabras antes vs despues", fontweight="bold")
axes[2].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Ejemplos concretos de cambios
print("EJEMPLOS DE NORMALIZACION")
print("=" * 70)

examples_shown = 0
for sid in sessions:
    orig_segs = sessions[sid]["segments"]
    norm_segs = normalized_sessions[sid]
    
    # Buscar segmentos con diferencias (por timestamp aprox)
    norm_by_start = {seg["start_ms"]: seg for seg in norm_segs}
    
    for orig in orig_segs:
        if examples_shown >= 10:
            break
        norm = norm_by_start.get(orig["start_ms"])
        if norm and orig["text"].strip() != norm["text"].strip():
            ts = str(timedelta(milliseconds=orig["start_ms"]))[:-3]
            print(f"\n[{sid} {ts}]")
            print(f"  ANTES:   {orig['text'][:100]}")
            print(f"  DESPUES: {norm['text'][:100]}")
            examples_shown += 1
    if examples_shown >= 10:
        break

if examples_shown == 0:
    print("No se encontraron diferencias en segmentos con timestamps coincidentes.")
    print("(Las diferencias principales son eliminacion de alucinaciones y cuasi-duplicados.)")

---
## 13. Tabla resumen por sesion

In [ ]:
print(f"{'Sesion':<12s} {'Segs_IN':>8s} {'Segs_OUT':>9s} {'Hallu':>6s} {'Dupes':>6s} {'Terms':>6s} {'Punct':>6s} {'Total':>6s}")
print("-" * 70)

grand_total = 0
for sid in sorted(all_stats.keys()):
    st = all_stats[sid]
    total = st["hallucinations_removed"] + st["near_dupes_merged"] + st["terms_corrected"] + st["punct_normalized"]
    grand_total += total
    print(f"{sid:<12s} {st['input_segments']:>8d} {st['output_segments']:>9d} "
          f"{st['hallucinations_removed']:>6d} {st['near_dupes_merged']:>6d} "
          f"{st['terms_corrected']:>6d} {st['punct_normalized']:>6d} {total:>6d}")

print("-" * 70)
print(f"{'TOTAL':<12s} {total_before:>8d} {total_after:>9d} "
      f"{total_hallu:>6d} {total_dupes:>6d} "
      f"{total_terms:>6d} {total_punct:>6d} {grand_total:>6d}")

In [ ]:
# --- Grafico 10: Heatmap de normalizacion + curva de crecimiento de vocabulario ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Heatmap de acciones de normalizacion por sesion
action_keys = ["hallucinations_removed", "near_dupes_merged", "terms_corrected",
               "unicode_normalized", "chars_cleaned", "punct_normalized"]
action_names = ["Alucinac.", "Duplicados", "Terminos", "Unicode", "Chars", "Puntuacion"]

heatmap_data = np.array([[all_stats[s][k] for k in action_keys] for s in sids])
session_labels = [s.replace("llm_", "") for s in sids]

im = axes[0].imshow(heatmap_data, cmap="YlOrRd", aspect="auto")
axes[0].set_xticks(range(len(action_names)))
axes[0].set_xticklabels(action_names, rotation=45, ha="right", fontsize=10)
axes[0].set_yticks(range(len(session_labels)))
axes[0].set_yticklabels(session_labels, fontsize=10)
axes[0].set_title("Heatmap: acciones por sesion", fontweight="bold")

# Anotar valores
for i in range(len(sids)):
    for j in range(len(action_keys)):
        val = heatmap_data[i, j]
        if val > 0:
            color = "white" if val > heatmap_data.max() * 0.6 else "black"
            axes[0].text(j, i, str(int(val)), ha="center", va="center", fontsize=9, color=color)

fig.colorbar(im, ax=axes[0], shrink=0.8)

# Curva de crecimiento de vocabulario (Heaps' law)
vocab_growth = []
seen = set()
word_count = 0
checkpoint = 500

all_words_ordered = []
for sid in sorted(sessions.keys()):
    for seg in sessions[sid]["segments"]:
        all_words_ordered.extend(seg["text"].lower().split())

for i, word in enumerate(all_words_ordered):
    seen.add(word)
    word_count += 1
    if word_count % checkpoint == 0:
        vocab_growth.append((word_count, len(seen)))

if vocab_growth:
    wc, vc = zip(*vocab_growth)
    axes[1].plot(wc, vc, color=PALETTE[0], linewidth=2, label="Vocabulario real")

    # Ajuste Heaps: V = K * N^beta
    log_wc = np.log(np.array(wc))
    log_vc = np.log(np.array(vc))
    beta, log_k = np.polyfit(log_wc, log_vc, 1)
    K = np.exp(log_k)
    heaps_fit = K * np.array(wc) ** beta
    axes[1].plot(wc, heaps_fit, "--", color=PALETTE[6], linewidth=1.5,
                 label=f"Heaps' law (K={K:.1f}, β={beta:.2f})")

    axes[1].set_xlabel("Palabras procesadas")
    axes[1].set_ylabel("Vocabulario unico")
    axes[1].set_title("Crecimiento del vocabulario (Heaps' law)", fontweight="bold")
    axes[1].legend()
    axes[1].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

plt.tight_layout()
plt.show()

---
## 14. Exportar transcripciones normalizadas

In [ ]:
saved = 0
for sid, norm_segs in normalized_sessions.items():
    s = sessions[sid]
    data = deepcopy(s["data"])
    
    # Actualizar segmentos
    audio_key = "audio" if "audio" in data else None
    if audio_key:
        data[audio_key]["segments"] = norm_segs
        # Regenerar full_text
        data[audio_key]["full_text"] = " ".join(seg["text"] for seg in norm_segs)
        # Agregar metadata de normalizacion
        data[audio_key]["normalization"] = all_stats[sid]
    else:
        data["segments"] = norm_segs
        data["full_text"] = " ".join(seg["text"] for seg in norm_segs)
        data["normalization"] = all_stats[sid]
    
    # Guardar
    out_path = s["path"].parent / "audio_normalized.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    size_kb = out_path.stat().st_size / 1024
    print(f"  {sid}: {out_path.name} ({size_kb:.0f} KB, {len(norm_segs)} segmentos)")
    saved += 1

print(f"\nExportados: {saved} archivos audio_normalized.json")

---
## 15. Resumen final

In [ ]:
print("=" * 70)
print("NOTEBOOK 02: NORMALIZACION COMPLETADO")
print("=" * 70)
print(f"")
print(f"Analisis realizado:")
print(f"  - Alucinaciones (repeticiones): {total_hallucinated_segs} segmentos")
print(f"  - Cuasi-duplicados: {total_near_dupes} pares")
print(f"  - Errores ASR en terminos tecnicos: {sum(error_counts.values())} ocurrencias")
print(f"  - Mezcla de idiomas: {lang_stats.get('MIXED', 0)} segmentos mixtos")
print(f"  - Problemas de puntuacion y capitalizacion")
print(f"  - Segmentos anomalos (cortos/largos/caracteres)")
print(f"")
print(f"Normalizacion aplicada:")
print(f"  Segmentos: {total_before:,} -> {total_after:,} (-{total_before - total_after:,})")
print(f"  Palabras: {words_before:,} -> {words_after:,}")
print(f"  Total acciones: {total_actions:,}")
print(f"")
print(f"Output: artifacts/llm_*/audio_normalized.json")
print(f"")
print(f"Proximo paso: Notebook 03 (Pseudo-Label Generation)")
print("=" * 70)